# Notebook 02 - Preprocesamiento y Feature Engineering

**MatchMind: Prediccion de Resultados de Futbol Internacional**

---

## Por que este notebook es CRITICO

El feature engineering suele ser **donde se gana o se pierde un proyecto de Machine Learning**. Si las features son malas, ningun algoritmo (ni el mas sofisticado) va a funcionar bien. Si las features son buenas, hasta una regresion logistica simple obtiene buenos resultados.

En este notebook construimos 14 features que capturan la informacion predictiva del problema:

1. **Racha reciente** de cada equipo (victorias en los ultimos 5 partidos).
2. **Promedios de goles** anotados y recibidos (ventana de 10 partidos).
3. **Head-to-head historico** (victorias previas entre ambos equipos).
4. **Ranking FIFA y puntos** de cada equipo en el momento del partido.
5. **Diferencia de ranking** y **diferencia de puntos**.
6. **Neutralidad del campo**.

## El reto: evitar DATA LEAKAGE

Data leakage es el error mas peligroso en ML: cuando una feature contiene informacion del futuro o del propio target. Resultado: el modelo da metricas espectaculares en validacion pero falla en produccion.

Ejemplo concreto en nuestro caso: si al calcular ``racha reciente`` del partido del 1 de junio incluimos el resultado del propio partido del 1 de junio, el modelo "adivinaria" el resultado mirando una feature que en realidad no podriamos calcular antes del partido.

Solucion: usamos `groupby().rolling().shift(1)` para garantizar que cada feature solo vea informacion ESTRICTAMENTE anterior al partido actual.

## 0. Imports

Importamos numpy, pandas y nuestras funciones de `src/`. La funcion clave es `build_features()` que orquesta todo el pipeline vectorizado.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np

from src.data.load import load_results, load_fifa_ranking, make_target
from src.data.features import build_features
from src.models.train import temporal_split, get_feature_columns

print('Imports OK. Nota: train.py ahora hace import lazy de xgboost,')
print('asi que este notebook funciona aunque no tengas xgboost instalado.')

Imports OK. Nota: train.py ahora hace import lazy de xgboost,
asi que este notebook funciona aunque no tengas xgboost instalado.


## 1. Cargar y preparar

Cargamos el dataset, generamos la variable objetivo y eliminamos los nulos identificados en el notebook 01.

In [2]:
df = load_results()
df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)
df = make_target(df)
ranking = load_fifa_ranking()

print(f'Partidos: {len(df):,}')
print(f'Filas ranking FIFA: {len(ranking):,}')

Partidos: 49,215
Filas ranking FIFA: 67,472


## 2. Filtrar partidos posteriores a 1993

Recordatorio del notebook 01: el ranking FIFA empezo en 1992. Antes de eso no tenemos ranking confiable. Ademas, antes de 1950 casi todos los partidos eran europeos: el modelo no podria generalizar.

**Decision:** filtramos a partidos desde 1993-01-01. Pasamos de 49,215 a 30,511 partidos.

In [3]:
df = df[df['date'] >= '1993-01-01'].reset_index(drop=True)
print(f'Partidos post-1993: {len(df):,}')

Partidos post-1993: 30,511


## 3. Feature engineering vectorizado

La funcion `build_features()` (en `src/data/features.py`) implementa todo el pipeline. Internamente:

1. Convierte el dataframe a **formato largo** (un partido = dos filas, una por equipo).
2. Para cada equipo, calcula rolling window con `shift(1)` para evitar leakage.
3. Calcula head-to-head historico entre cada par de equipos.
4. Hace merge del ranking FIFA por fecha mas cercana anterior (con `merge_asof`).

Tiempo de ejecucion: alrededor de 20-25 segundos en una laptop estandar. Si usaramos `iterrows()` (la version ingenua), tardaria varias horas.

In [4]:
%%time
features = build_features(df, ranking_df=ranking, n_recent=5, window_goals=10)
print(f'\nShape final: {features.shape}')
print(f'Columnas generadas: {list(features.columns)}')


Shape final: (30511, 25)
Columnas generadas: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'result', 'home_recent_wins', 'home_goals_avg_10', 'home_goals_conceded_avg_10', 'away_recent_wins', 'away_goals_avg_10', 'away_goals_conceded_avg_10', 'h2h_home_wins', 'h2h_away_wins', 'h2h_draws', 'home_team_rank', 'home_team_points', 'away_team_rank', 'away_team_points', 'rank_diff', 'points_diff']
CPU times: total: 36.3 s
Wall time: 37.4 s


C:\Users\matia\OneDrive\Desktop\Matías M\EAFIT ING DE SISTEMAS\semestre 5\Inteligencia artificial\FINAL\matchmind-ia-eafit\src\data\features.py:165: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = g.apply(victories_for_team).reset_index(drop=True)


## 4. Imputacion de nulos del ranking

Hay ~3,700 partidos donde uno de los equipos no tenia ranking FIFA aun (equipo recien creado, primera aparicion, etc.). Tenemos que decidir que hacer con esos nulos.

**Opciones:**
- Eliminar esas filas: perderiamos un 12% del dataset.
- Imputar con la mediana: razonable, porque la mediana representa "un equipo promedio".
- Imputar con un valor especial (-1): los arboles lo pueden manejar pero anade complejidad.

**Decision:** imputar con la mediana del ranking (75).

In [5]:
print('Nulos antes de imputar:')
print(features[['home_team_rank', 'away_team_rank',
                'home_team_points', 'away_team_points']].isnull().sum())

med_rank = features['home_team_rank'].median()
print(f'\nMediana del rank usada para imputar: {med_rank}')

features['home_team_rank'] = features['home_team_rank'].fillna(med_rank)
features['away_team_rank'] = features['away_team_rank'].fillna(med_rank)
features['home_team_points'] = features['home_team_points'].fillna(0)
features['away_team_points'] = features['away_team_points'].fillna(0)

# Recalcular diferencias
features['rank_diff'] = features['home_team_rank'] - features['away_team_rank']
features['points_diff'] = features['home_team_points'] - features['away_team_points']
features['neutral'] = features['neutral'].astype(int)

print('\nNulos despues:')
print(features[['home_team_rank', 'away_team_rank']].isnull().sum())

Nulos antes de imputar:
home_team_rank      3717
away_team_rank      3761
home_team_points    3714
away_team_points    3755
dtype: int64

Mediana del rank usada para imputar: 75.0

Nulos despues:
home_team_rank    0
away_team_rank    0
dtype: int64


## 5. Guardar el dataset procesado

Guardamos `features.csv` para reutilizar en los siguientes notebooks. Esto es buena practica: si algo falla en el modeling, no tenemos que recalcular el feature engineering.

In [6]:
out_path = Path('../data/processed/features.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(out_path, index=False)
print(f'Guardado: {out_path}  ({out_path.stat().st_size / 1024 / 1024:.1f} MB)')

Guardado: ..\data\processed\features.csv  (4.3 MB)


## 6. Split temporal - el detalle critico

**Nunca usamos `train_test_split` aleatorio en datos temporales.** Esto creariia data leakage temporal: el modelo veria partidos de 2024 en entrenamiento y luego se probaria con partidos de 2020, lo cual nunca ocurriria en produccion.

**Split correcto:**
- **Train:** 1993-2015 (los primeros ~70%)
- **Val:** 2016-2019 (~14%) - usado para tuning de hiperparametros
- **Test:** 2020-2026 (~16%) - evaluacion final, una sola pasada

Esto simula el escenario real: entrenamos con el pasado, predecimos el futuro.

In [7]:
train, val, test = temporal_split(features,
                                  train_end='2015-12-31',
                                  val_end='2019-12-31')

print(f'Train: {len(train):,} partidos  ({train["date"].min().date()} -> {train["date"].max().date()})')
print(f'Val:   {len(val):,} partidos   ({val["date"].min().date()} -> {val["date"].max().date()})')
print(f'Test:  {len(test):,} partidos  ({test["date"].min().date()} -> {test["date"].max().date()})')

Train: 20,715 partidos  (1993-01-01 -> 2015-12-31)
Val:   3,920 partidos   (2016-01-03 -> 2019-12-29)
Test:  5,876 partidos  (2020-01-07 -> 2026-03-31)


## 7. Seleccion final de columnas y guardado del split

De las 24 columnas que tiene `features`, solo 14 son features numericas para el modelo. Las otras son metadata (nombres de equipos, fechas, scores). Usamos `get_feature_columns()` que devuelve exactamente las 14 features.

Guardamos todo en un pickle para que los notebooks 03 y 04 lo carguen directamente sin reprocesar.

In [8]:
import pickle

feature_cols = get_feature_columns()
print(f'Features finales ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  - {c}')

split = {
    'X_train': train[feature_cols].astype(float),
    'y_train': train['result'].values,
    'X_val':   val[feature_cols].astype(float),
    'y_val':   val['result'].values,
    'X_test':  test[feature_cols].astype(float),
    'y_test':  test['result'].values,
    'meta_train': train[['date', 'home_team', 'away_team', 'tournament']].reset_index(drop=True),
    'meta_val':   val[['date', 'home_team', 'away_team', 'tournament']].reset_index(drop=True),
    'meta_test':  test[['date', 'home_team', 'away_team', 'tournament']].reset_index(drop=True),
}

with open('../data/processed/train_test_split.pkl', 'wb') as f:
    pickle.dump(split, f)
print('\nSplit guardado en data/processed/train_test_split.pkl')

Features finales (14):
  - home_recent_wins
  - away_recent_wins
  - home_goals_avg_10
  - home_goals_conceded_avg_10
  - away_goals_avg_10
  - away_goals_conceded_avg_10
  - h2h_home_wins
  - h2h_draws
  - h2h_away_wins
  - home_team_rank
  - away_team_rank
  - rank_diff
  - points_diff
  - neutral

Split guardado en data/processed/train_test_split.pkl


## 8. Sanity check

Antes de pasar al modeling, verificamos:
1. Las distribuciones de clase en train y test son similares (no hay drift fuerte).
2. No hay nulos en ninguno de los splits.
3. Las features tienen rangos razonables.

In [9]:
print('Distribucion de clases en TRAIN:')
print(pd.Series(split['y_train']).value_counts(normalize=True).sort_index().rename({0: 'away', 1: 'draw', 2: 'home'}))
print('\nDistribucion de clases en TEST:')
print(pd.Series(split['y_test']).value_counts(normalize=True).sort_index().rename({0: 'away', 1: 'draw', 2: 'home'}))

print('\nNulos en splits:')
for name in ['X_train', 'X_val', 'X_test']:
    print(f'  {name}: {split[name].isnull().sum().sum()} nulos')

Distribucion de clases en TRAIN:
away    0.277239
draw    0.233454
home    0.489307
Name: proportion, dtype: float64

Distribucion de clases en TEST:
away    0.294078
draw    0.231450
home    0.474472
Name: proportion, dtype: float64

Nulos en splits:
  X_train: 0 nulos
  X_val: 0 nulos
  X_test: 0 nulos


**Distribucion estable** entre train y test (~49%/23%/28%). No hay drift fuerte. 0 nulos en todos los splits.

## Siguiente paso

Notebook **03_modeling.ipynb**: entrenamos 4 modelos (Dummy, LogReg, Random Forest, XGBoost), los comparamos y aplicamos SHAP.